# CCM-109 - Tópicos especiais de IA - Deep Learning

## Pipeline de extração de rostos por frame

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/main.ipynb)

---

### Índice

1. [Configuração](#1-configuração)
2. [Funções auxiliares](#2-funções-auxiliares)
3. [Execução pipeline](#3-execução-da-pipeline)

## 1. Configuração

In [57]:
# @title Instalar dependências

%pip install deepface opencv-python tqdm tf-keras supervision

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 63.7 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 54.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 54.7 MB/s  0:00:00 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [supervision]7m━━━━ 8/9 [supervision]
Note: you may need to restart the kernel to use updated packages.


In [58]:
# @title Imports

import random
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import supervision as sv
from tqdm import tqdm
from deepface import DeepFace

In [ ]:
# @title Configurações do pipeline

VIDEO_PATH = "/dataset/"
OUTPUT_DIR = "/dataset/"

DETECTOR_BACKEND = "retinaface"      
CONFIDENCE_THRESHOLD = 0.9           
ALIGN = True                          

SKIP_FRAMES = 10                    # 0 = todos; 4 = pula 4 (processa 1 a cada 5)
MAX_FRAMES_PER_VIDEO = 50          # None = todos; ex: 100 = máx 100 frames salvos
MAX_READ_LIMIT = 300                    # None = todos; ex: 1000 = máx 1000 frames lidos
MAX_VIDEOS = 5                    # None = todos; ex: 10 = máx 10 vídeos processados 

VIDEO_EXTENSIONS = ("*.mp4", "*.avi", "*.mov", "*.mkv")

tracker = sv.ByteTrack()

## 2. Funções auxiliares

In [60]:
# @title Extrair rostos de um frame

def extract_faces_from_frame(frame, detector_backend="retinaface", 
                             confidence_threshold=0.85, align=True):
    
    try:
        detected_faces = DeepFace.extract_faces(
            img_path=frame,
            detector_backend=detector_backend,
            enforce_detection=False,
            align=align
        )
    except Exception:
        return []

    boxes, confidences, valid_face_imgs = [], [], []

    for face_info in detected_faces:
        conf = face_info.get("confidence", 0)
        if conf >= confidence_threshold:
            area = face_info.get("facial_area", {})
            x, y, w, h = area.get("x", 0), area.get("y", 0), area.get("w", 0), area.get("h", 0)
            
            boxes.append([x, y, x + w, y + h])
            confidences.append(conf)

            face_img = face_info["face"]
            if face_img.dtype in (np.float32, np.float64):
                face_img = (face_img * 255).astype(np.uint8)
            
            valid_face_imgs.append(face_img)

    if not boxes:
        return []

    detections = sv.Detections(
        xyxy=np.array(boxes),
        confidence=np.array(confidences)
    )

    tracked_detections = tracker.update_with_detections(detections)
    output_faces = []

    if tracked_detections.tracker_id is not None:
        for idx, tracker_id in enumerate(tracked_detections.tracker_id):
            if idx < len(valid_face_imgs):
                output_faces.append({
                    "face": valid_face_imgs[idx],
                    "face_id": int(tracker_id)
                })

    return output_faces

In [ ]:
# @title Processar vídeo

def process_video(video_path, output_dir, 
                             detector="retinaface",
                             confidence=0.85,
                             align=True,
                             skip_frames=0,
                             max_frames=30,
                             max_read_limit=None):
    
    global tracker
    tracker = sv.ByteTrack()

    video_path = Path(video_path)
    video_name = video_path.stem

    frames_dir = Path(output_dir) / video_name
    frames_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0

    frame_idx = 0
    saved_frames_count = 0

    while True:
        if max_frames is not None and saved_frames_count >= max_frames:
            break
            
        if max_read_limit is not None and frame_idx >= max_read_limit:
            break

        ret, frame = cap.read()
        if not ret:
            break

        if skip_frames > 0 and frame_idx % (skip_frames + 1) != 0:
            frame_idx += 1
            continue

        faces = extract_faces_from_frame(
            frame, 
            detector_backend=detector,
            confidence_threshold=confidence, 
            align=align
        )

        if faces:
            for face_data in faces:
                face_id = face_data["face_id"]
                filename = f"frame_{frame_idx + 1}_face_{face_id}.jpg"
                output_path = frames_dir / filename
                
                face_bgr = cv2.cvtColor(face_data["face"], cv2.COLOR_RGB2BGR)
                cv2.imwrite(str(output_path), face_bgr)
                
            saved_frames_count += 1

        frame_idx += 1

    cap.release()
    return saved_frames_count

In [62]:
# @title Processar o dataset

def process_dataset(dataset_root, max_videos_per_folder=5, seed=None):
    dataset_root = Path(dataset_root)

    if seed is not None:
        random.seed(seed)

    video_files = []
    for ext in VIDEO_EXTENSIONS:
        video_files.extend(list(dataset_root.rglob(ext)))

    if not video_files:
        return

    videos_by_folder = defaultdict(list)
    for video in video_files:
        videos_by_folder[video.parent].append(video)

    total_frames_dataset = 0
    total_videos_processed = 0

    for folder, files in videos_by_folder.items():
        if max_videos_per_folder and len(files) > max_videos_per_folder:
            videos_to_process = random.sample(files, max_videos_per_folder)
        else:
            videos_to_process = files.copy()
            random.shuffle(videos_to_process)

        if folder.name == "videos":
            video_output_dir = folder.parent / "frames"
        else:
            video_output_dir = folder / "frames"

        pbar = tqdm(videos_to_process, desc=f"Processando {folder.name}", unit="video")

        for video_path in pbar:
            pbar.set_postfix({"Vídeo": video_path.name[:20]})

            saved_frames = process_video(
                video_path=video_path,
                output_dir=video_output_dir,
                confidence=CONFIDENCE_THRESHOLD,
                align=ALIGN,
                skip_frames=SKIP_FRAMES,
                max_frames=MAX_FRAMES_PER_VIDEO,
                max_read_limit=MAX_READ_LIMIT
            )

            total_frames_dataset += saved_frames
            total_videos_processed += 1

    print("\n" + "=" * 65)
    print(f"🎉 PROCESSAMENTO CONCLUÍDO!")
    print(f"   Total de pastas varridas: {len(videos_by_folder)}")
    print(f"   Total de vídeos processados: {total_videos_processed}")
    print(f"   Total de frames salvos: {total_frames_dataset}")
    print("=" * 65)

## 3. Execução da pipeline

In [63]:
# @title Executar pipeline de extração de rostos

stats = process_dataset(
    dataset_root=VIDEO_PATH,
    max_videos_per_folder=MAX_VIDEOS,
    seed=None
)



Processando videos: 100%|██████████| 5/5 [05:20<00:00, 64.01s/video, Vídeo=803_017.mp4]


🎉 PROCESSAMENTO CONCLUÍDO!
   Total de pastas varridas: 8
   Total de vídeos processados: 40
   Total de frames salvos: 1115
